# Student Academic Risk Assessment Model

## Purpose
Train a **LightGBM** classifier to estimate **Low**, **Medium**, or **High** academic risk for each manually entered student/module record. The supplied labeled academic dataset is used for training; manual input rows are only scored after model evaluation and are never included in training.

## Improvements over the previous RandomForest model
- **LightGBM** instead of RandomForest — gradient-boosted trees generalise better on tabular data with class imbalance.
- **Feature engineering** — five derived features (score average, GPA×attendance, progress gap, late submission ratio, performance index) amplify the natural separation between risk classes.
- **Automatic balanced class weights** — computed from the training-set class distribution, so the minority High-risk class receives proportionally more attention during training.

## Training practices
- Validate required columns, labels, and numeric feature values before fitting.
- Split data with stratification so each risk level is represented in both sets.
- Keep imputation and classification in one pipeline to avoid preprocessing leakage.
- Evaluate on held-out data before refitting the final model on all labeled examples.
- Save the complete pipeline to `python/RiskAssessment/ModelOutput/risk_model.joblib`.

Run this notebook in VS Code with the Python environment from `python/.venv` selected.

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================

from collections import Counter
from pathlib import Path
import warnings
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# Prefer the notebook folder; fallback supports VS Code kernels launched
# from the repository root, python folder, or another current directory.
HERE = Path.cwd().resolve()
candidates = [HERE, *HERE.parents]
RISK_ASSESSMENT_DIR = next(
    (p for p in candidates if p.name == "RiskAssessment" and (p / "model.py").is_file()),
    None,
)
if RISK_ASSESSMENT_DIR is None:
    RISK_ASSESSMENT_DIR = next(
        (p / "RiskAssessment" for p in candidates if (p / "RiskAssessment" / "model.py").is_file()),
        None,
    )
if RISK_ASSESSMENT_DIR is None:
    raise FileNotFoundError("Could not find python/RiskAssessment/model.py from the current directory.")

PYTHON_DIR = RISK_ASSESSMENT_DIR.parent
DATA_DIR = PYTHON_DIR / "data"
TRAINING_FILE = DATA_DIR / "student_academic_risk.csv"
MANUAL_FILE = DATA_DIR / "manual_risk_inputs.csv"
MODEL_DIR = RISK_ASSESSMENT_DIR / "ModelOutput"
MODEL_FILE = MODEL_DIR / "risk_model.joblib"

# Base features from the training CSV
FEATURES = [
    "previous_gpa", "attendance", "assignment_score", "quiz_score",
    "study_hours", "engagement_score", "late_assignments",
    "completed_assignments", "learning_progress",
]
# Engineered features added during preprocessing
ENGINEERED_FEATURES = [
    "score_avg",          # mean of assignment + quiz scores
    "gpa_x_attendance",  # GPA scaled by attendance rate
    "progress_gap",       # learning_progress minus score_avg
    "late_ratio",         # late_assignments / completed_assignments
    "performance_index",  # score_avg * attendance_rate * progress_rate
]
MODEL_FEATURES = FEATURES + ENGINEERED_FEATURES
TARGET = "academic_risk_level"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("=" * 72)
print("ACADEMIC RISK MODEL CONFIGURATION")
print("=" * 72)
print(f"Training data  : {TRAINING_FILE}")
print(f"Manual inputs  : {MANUAL_FILE}")
print(f"Model output   : {MODEL_FILE}")
print(f"Test fraction  : {TEST_SIZE:.0%}")
print(f"Random seed    : {RANDOM_STATE}")
print(f"Base features  : {len(FEATURES)}")
print(f"Total features : {len(MODEL_FEATURES)} (incl. {len(ENGINEERED_FEATURES)} engineered)")


In [ ]:
# ============================================================
# LOAD AND EXPLORE DATA
# ============================================================

if not TRAINING_FILE.is_file():
    raise FileNotFoundError(
        f"Training CSV not found: {TRAINING_FILE}. Place student_academic_risk.csv in python/data/."
    )
data = pd.read_csv(TRAINING_FILE)
print(f"Loaded {len(data):,} rows and {len(data.columns)} columns.")
print("Columns:", list(data.columns))
print("\nPreview:")
display(data.head())
print("\nRisk label distribution (including missing labels):")
display(
    data["academic_risk_level"].value_counts(dropna=False)
    if "academic_risk_level" in data else "Target column missing"
)
print("\nMissing values in model columns:")
display(data.reindex(columns=FEATURES + [TARGET]).isna().sum())


## Check the training data
The target is `academic_risk_level`. Student identifiers are excluded from features because they identify a student rather than describe learning performance. The notebook preserves valid extreme values; unusual records should be investigated against the data source before removal.

In [ ]:
# ============================================================
# VALIDATE AND PREPARE BASE FEATURES
# ============================================================

missing_columns = [c for c in FEATURES + [TARGET] if c not in data.columns]
if missing_columns:
    raise ValueError(f"Training CSV is missing required columns: {missing_columns}")

label_map = {"low": "Low", "medium": "Medium", "high": "High"}
raw_labels = data[TARGET].astype("string").str.strip().str.lower()
labels = raw_labels.map(label_map)
unknown = sorted(raw_labels[labels.isna() & raw_labels.notna()].unique().tolist())
if unknown:
    raise ValueError(f"Unexpected risk labels in {TARGET}: {unknown}")
if labels.isna().any():
    n_missing = int(labels.isna().sum())
    raise ValueError(f"Target {TARGET} contains {n_missing} missing label(s).")

X_base = data[FEATURES].apply(pd.to_numeric, errors="coerce")
for column in FEATURES:
    invalid = data[column].notna() & X_base[column].isna()
    if invalid.any():
        raise ValueError(f"Feature {column!r} has {int(invalid.sum())} non-numeric value(s).")
y = labels
class_counts = y.value_counts()
if len(class_counts) != 3:
    raise ValueError(f"Expected all 3 risk levels; found: {class_counts.to_dict()}")
if class_counts.min() < 2:
    raise ValueError("Each risk level needs at least two rows for a stratified split.")

print(f"Usable rows: {len(X_base):,}")
print("Class distribution:")
display(class_counts.rename_axis("risk_level").to_frame("rows"))


In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
# These derived columns capture interaction effects that plain base features
# cannot express on their own, and meaningfully increase class separation.

def add_engineered_features(df):
    df = df.copy()
    df["score_avg"]         = (df["assignment_score"] + df["quiz_score"]) / 2
    df["gpa_x_attendance"]  = df["previous_gpa"] * df["attendance"] / 100
    df["progress_gap"]      = df["learning_progress"] - df["score_avg"]
    df["late_ratio"]        = df["late_assignments"] / df["completed_assignments"].clip(lower=1)
    df["performance_index"] = df["score_avg"] * df["attendance"] / 100 * df["learning_progress"] / 100
    return df

data_eng = add_engineered_features(X_base.join(y.rename("risk_level")))
X = data_eng[MODEL_FEATURES]

print("Engineered feature means by risk level:")
display(data_eng[ENGINEERED_FEATURES + ["risk_level"]].groupby("risk_level").mean())


In [ ]:
# ============================================================
# STRATIFIED TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows    : {len(X_test):,}")
print("\nTraining label proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))
print("Test label proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion"))


In [ ]:
# ============================================================
# TRAIN AND EVALUATE LIGHTGBM
# ============================================================

# Compute balanced class weights from training distribution
counts = Counter(y_train)
total = len(y_train)
class_weight = {cls: total / (3 * cnt) for cls, cnt in counts.items()}
print("Computed class weights:", class_weight)

def make_model(class_weight=None):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", lgb.LGBMClassifier(
            n_estimators=600,
            learning_rate=0.03,
            max_depth=10,
            num_leaves=127,
            min_child_samples=5,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )),
    ])

evaluation_model = make_model(class_weight=class_weight)
evaluation_model.fit(X_train, y_train)
y_pred = evaluation_model.predict(X_test)
y_train_pred = evaluation_model.predict(X_train)

accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 72)
print("HELD-OUT EVALUATION")
print("=" * 72)
print(f"Training accuracy : {train_accuracy:.4f}")
print(f"Test accuracy     : {accuracy:.4f}")
print(f"Macro F1          : {macro_f1:.4f} (each risk level has equal weight)")
print(f"Weighted F1       : {weighted_f1:.4f}")
print("\nPer-class report:")
print(classification_report(y_test, y_pred, labels=["Low", "Medium", "High"], zero_division=0))


In [ ]:
# ============================================================
# VISUALIZE HELD-OUT RESULTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=["Low", "Medium", "High"],
    cmap="Blues", values_format="d", ax=axes[0]
)
axes[0].set_title("Held-out student risk classification")

classifier = evaluation_model.named_steps["classifier"]
importance = pd.Series(classifier.feature_importances_, index=MODEL_FEATURES).sort_values()
importance.plot.barh(ax=axes[1], color="teal", title="LightGBM feature importance")
axes[1].set_xlabel("Importance")
axes[1].set_ylabel("Input feature")

plt.tight_layout()
plt.show()


## Refit on all labeled rows and save
After reviewing the held-out evaluation, train a fresh pipeline using all labeled training rows. The saved artifact includes both the median imputer and the LightGBM classifier, so the API can pass raw feature columns through the same preprocessing steps.

In [ ]:
# ============================================================
# TRAIN FINAL MODEL ON ALL DATA AND SAVE ARTIFACT
# ============================================================

counts_all = Counter(y)
total_all = len(y)
class_weight_all = {cls: total_all / (3 * cnt) for cls, cnt in counts_all.items()}

model = make_model(class_weight=class_weight_all)
model.fit(X, y)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_FILE)
if not MODEL_FILE.is_file():
    raise OSError(f"Model file was not created: {MODEL_FILE}")

final_train_acc = accuracy_score(y, model.predict(X))

print("=" * 72)
print("FINAL MODEL SAVED")
print("=" * 72)
print(f"Path           : {MODEL_FILE.resolve()}")
print(f"Size           : {MODEL_FILE.stat().st_size:,} bytes")
print(f"Train rows     : {len(X):,}")
print(f"Train accuracy : {final_train_acc:.4f}")
print("Classes        :", list(model.classes_))


In [ ]:
# ============================================================
# SCORE MANUAL STUDENT / MODULE INPUTS (OPTIONAL)
# ============================================================

if not MANUAL_FILE.is_file():
    print(f"Manual input CSV not found; skipping preview: {MANUAL_FILE}")
else:
    manual = pd.read_csv(MANUAL_FILE)
    required_manual = ["user_id", "student_id", "student_name", "module_id", "module_name"] + FEATURES
    missing_manual = [c for c in required_manual if c not in manual.columns]
    if missing_manual:
        raise ValueError(f"Manual input CSV is missing columns: {missing_manual}")
    if manual.empty:
        print("Manual input file has no student/module rows.")
    else:
        manual_X = manual[FEATURES].apply(pd.to_numeric, errors="coerce")
        manual_eng = add_engineered_features(manual_X)
        manual["risk_level"] = model.predict(manual_eng[MODEL_FEATURES])
        manual["confidence"] = model.predict_proba(manual_eng[MODEL_FEATURES]).max(axis=1)
        shown = [
            "user_id", "student_id", "student_name", "module_id", "module_name",
            "risk_level", "confidence", "attendance", "assignment_score",
            "quiz_score", "engagement_score", "learning_progress",
        ]
        print("Predicted risk for each manually entered module (preview only):")
        display(manual[shown])
        at_risk = manual[manual["risk_level"].isin(["High", "Medium"])]
        print(f"{len(at_risk)} High/Medium risk module(s) from {len(manual)} manual row(s).")

print("\nReview the printed paths to find the input CSV and saved artifact.")
